# Détecteur de points — grille MINOR (1mm)

Variante de `training_npz_Gaussienne` ciblant la **grille minor 1mm** (`grid_minor_1mm`) au lieu de la 5mm.

**Deux changements vs la major :**
1. `npz_key = grid_minor_1mm` (sortie : `runs_npz_gaussienne_minor/`).
2. `_draw_points_mask` **vectorisé** (splat + flou gaussien) — obligatoire car la minor a ~**60 000 points/image** ; la boucle Python ferait ~1 s/image (~34 min/epoch rien que pour les masques). Vectorisé = ~87 ms/image.

⚠️ **Densité** : heatmap bien plus dense (~11 % de couverture vs 0.45 % en major), pics de ~1-2 px très rapprochés à 1024². Le Dice sera plus haut (plus de foreground) sans que ça signifie « mieux ». Si les blobs fusionnent, baisser `gaussian_sigma`.


In [ ]:
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "9.0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [ ]:
import os, sys, time, json, datetime, argparse
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
torch.backends.cudnn.benchmark = False

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from PIL import Image

import cv2
import segmentation_models_pytorch as smp
import albumentations as A

# Acces au schema NPZ partage du projet
PROJECT_ROOT = r"C:\Users\v\Desktop\ECGPerturb-main"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from shared.npz_schema import load_unified_npz, load_unified


In [ ]:
from dataclasses import dataclass, field

@dataclass
class TrainConfig:
    # Configuration d'entrainement pour la detection de points d'intersection (1mm) via NPZ.

    # Chemins
    project_root: str = r"C:\Users\v\Desktop\ECGPerturb-main\data"
    image_dir: str = ""
    npz_dir:   str = ""
    output_dir: str = ""

    # Cible : intersections de la grille 1mm (cle NPZ)
    npz_key: str = "grid_minor_1mm"
    point_radius: int = 5         # rayon (px) de la fenetre dans laquelle la gaussienne est dessinee
    gaussian_sigma: float = 2.0   # ecart-type de la gaussienne 2D (px). Pic au centre = 1.0, decroit vers les bords.

    # Architecture
    encoder_name: str = "resnet34"
    encoder_weights: str = "imagenet"
    in_channels: int = 3
    num_classes: int = 1

    # Resolution
    img_height: int = 1024
    img_width:  int = 1024

    # Entrainement
    batch_size: int = 4
    num_epochs: int = 100
    learning_rate: float = 1e-4
    weight_decay:  float = 1e-5
    num_workers: int = 0
    pin_memory:  bool = True

    # Loss & scheduler
    loss_type: str = "bce_dice"
    bce_weight: float = 0.5
    scheduler_patience: int = 5
    scheduler_factor: float = 0.5
    early_stop_patience: int = 15

    # Split par prefixe de nom de fichier
    train_sources: list = field(default_factory=lambda: ["ECG_031", "ECG_032"])
    val_sources:   list = field(default_factory=lambda: ["ECG_033"])

    device: str = ""
    seed: int = 42
    save_every_n_epochs:    int = 10
    log_images_every_n_epochs: int = 5

    def __post_init__(self):
        if not self.image_dir:
            self.image_dir = os.path.join(self.project_root, "output_augmentation", "images")
        if not self.npz_dir:
            self.npz_dir = os.path.join(self.project_root, "output_augmentation", "labels")
        if not self.output_dir:
            self.output_dir = os.path.join(self.project_root, "training", "runs_npz_gaussienne_minor")
        if not self.device:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"

        if self.device == "cuda":
            print(f"[OK] GPU detecte : {torch.cuda.get_device_name(0)}")
            vram = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"   VRAM: {vram:.1f} GB")
        else:
            print("[!] Mode CPU actif - performances limitees a 1024x1024.")

cfg = TrainConfig()


In [ ]:
class ECGNpzDataset(Dataset):
    # Dataset qui apparie chaque image augmentee P2 avec un mask binaire
    # genere a la volee depuis le NPZ (points d'intersection grille minor).
    # Le mask est dessine en pleine resolution image (W,H native du NPZ),
    # puis redimensionne (NEAREST) a (img_width, img_height).

    def __init__(self, image_dir, npz_dir, npz_key="grid_minor_1mm",
                 source_prefixes=None, img_height=1024, img_width=1024,
                 point_radius=3, gaussian_sigma=2.0, augment=False):
        self.image_dir = image_dir
        self.npz_dir   = npz_dir
        self.npz_key   = npz_key
        self.img_height, self.img_width = img_height, img_width
        self.point_radius   = point_radius     # fenetre (rayon) ou la gaussienne est dessinee
        self.gaussian_sigma = gaussian_sigma   # ecart-type de la gaussienne (px)

        self.samples = []
        for fname in sorted(os.listdir(image_dir)):
            if not fname.endswith(".webp"):
                continue
            if source_prefixes and not any(fname.startswith(p) for p in source_prefixes):
                continue
            stem = os.path.splitext(fname)[0]
            npz_path = os.path.join(npz_dir, stem)
            labels_file = os.path.join(npz_path, "labels.npy.zst")
            if os.path.isfile(labels_file) and os.path.getsize(labels_file) > 0:
                self.samples.append({
                    "image_path": os.path.join(image_dir, fname),
                    "npz_path":   npz_path,
                    "stem":       stem,
                })

        # Augmentations couleur uniquement (la geometrie casserait l'alignement points/image)
        if augment:
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.3),
                A.ColorJitter(brightness=0.15, contrast=0.15,
                              saturation=0.1, hue=0.05, p=0.5),
                A.GaussNoise(p=0.2),
            ])
        else:
            self.transform = None

        print(f"  -> {len(self.samples)} paires trouvees (key: {npz_key},"
              f" sources: {source_prefixes or 'toutes'})")

    def __len__(self):
        return len(self.samples)

    @staticmethod
    def _draw_points_mask(pts_xy, H, W, radius, sigma=2.0):
        # Dessine des gaussiennes 2D centrees aux coordonnees pts_xy (Nx2) sur un fond noir HxW.
        # Chaque point recoit une gaussienne d'ecart-type 'sigma' (px), dessinee dans une fenetre
        # carree de rayon 'radius'. Pic au centre = 255 (apres normalisation), decroit progressivement.
        # Si plusieurs gaussiennes se chevauchent, on prend le MAX (pas la somme) pour eviter la
        # saturation et garder des pics independants.
        mask = np.zeros((H, W), dtype=np.float32)
        if len(pts_xy) == 0:
            return mask.astype(np.uint8)

        xs_f = pts_xy[:, 0]
        ys_f = pts_xy[:, 1]
        valid = (xs_f >= 0) & (xs_f < W) & (ys_f >= 0) & (ys_f < H)
        xs_f, ys_f = xs_f[valid], ys_f[valid]

        # VECTORISE (rapide ~11x) : splat des points + flou gaussien (O(image), pas O(points)).
        # Necessaire pour la grille minor (~60k points/image). Equivalent a la boucle (Dice 0.89),
        # le splat arrondit au pixel mais l'ecart sub-pixel (<0.3px natif) est negligeable apres resize.
        xs = np.clip(np.round(xs_f).astype(int), 0, W - 1)  # clip : round peut pousser <W a ==W (hors-borne)
        ys = np.clip(np.round(ys_f).astype(int), 0, H - 1)
        mask[ys, xs] = 1.0
        k = int(round(3 * sigma)) * 2 + 1
        mask = cv2.GaussianBlur(mask, (k, k), sigma)
        _ref = np.zeros((k + 4, k + 4), np.float32); _ref[(k + 4) // 2, (k + 4) // 2] = 1.0
        _peak1 = float(cv2.GaussianBlur(_ref, (k, k), sigma).max())  # pic d'une impulsion isolee -> normalise les pics a 1.0
        mask = (mask / max(_peak1, 1e-8)).clip(0, 1)
        return (mask * 255).clip(0, 255).astype(np.uint8)

    def __getitem__(self, idx):
        s = self.samples[idx]

        # Image
        img = Image.open(s["image_path"]).convert("RGB")
        Wn, Hn = img.size  # taille native (avant resize)
        img = img.resize((self.img_width, self.img_height), Image.BILINEAR)
        img_np = np.array(img, dtype=np.float32) / 255.0

        # Mask depuis NPZ
        data = load_unified(s["npz_path"], load_maps=False)
        pts = data.get(self.npz_key, np.empty((0, 2)))
        mask_full = self._draw_points_mask(pts, Hn, Wn, self.point_radius, self.gaussian_sigma)
        mask_pil = Image.fromarray(mask_full).resize(
            (self.img_width, self.img_height), Image.NEAREST
        )
        mask_np = np.array(mask_pil, dtype=np.float32) / 255.0

        # Augmentations couleur
        if self.transform:
            t = self.transform(image=img_np, mask=mask_np)
            img_np, mask_np = t["image"], t["mask"]

        img_tensor  = torch.from_numpy(img_np).permute(2, 0, 1).float()
        mask_tensor = torch.from_numpy(mask_np).unsqueeze(0).float()
        return img_tensor, mask_tensor


In [ ]:
# ═══════════════ Loss ═══════════════
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, pred, target):
        ps = torch.sigmoid(pred)
        inter = (ps * target).sum(dim=(2, 3))
        union = ps.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        return 1 - ((2 * inter + self.smooth) / (union + self.smooth)).mean()

class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce, self.dice, self.bce_weight = nn.BCEWithLogitsLoss(), DiceLoss(), bce_weight
    def forward(self, pred, target):
        return self.bce_weight * self.bce(pred, target) + (1 - self.bce_weight) * self.dice(pred, target)

def get_loss(loss_type, bce_weight=0.5):
    return {"bce": nn.BCEWithLogitsLoss(),
            "dice": DiceLoss(),
            "bce_dice": BCEDiceLoss(bce_weight)}[loss_type]


# ═══════════════ Metrics ═══════════════
def compute_metrics(pred, target, threshold=0.5):
    with torch.no_grad():
        pb = (torch.sigmoid(pred) > threshold).float()
        inter = (pb * target).sum(dim=(2, 3))
        ps = pb.sum(dim=(2, 3)); ts = target.sum(dim=(2, 3))
        dice = (2*inter + 1e-6) / (ps + ts + 1e-6)
        iou  = (inter + 1e-6) / (ps + ts - inter + 1e-6)
        rec  = (inter + 1e-6) / (ts + 1e-6)
        prec = (inter + 1e-6) / (ps + 1e-6)
    return {"dice": dice.mean().item(), "iou": iou.mean().item(),
            "precision": prec.mean().item(), "recall": rec.mean().item()}


# ═══════════════ Visualisation ═══════════════
def save_prediction_grid(images, masks_true, masks_pred, save_path, n=4):
    n = min(n, images.shape[0])
    fig, axes = plt.subplots(n, 4, figsize=(16, 4*n))
    if n == 1: axes = axes[np.newaxis, :]
    for i in range(n):
        img = images[i].cpu().permute(1, 2, 0).numpy()
        mt  = masks_true[i, 0].cpu().numpy()
        mp  = (torch.sigmoid(masks_pred[i, 0]).cpu().numpy() > 0.5).astype(float)
        axes[i, 0].imshow(img);                  axes[i, 0].set_title("Image augmentee", fontsize=9); axes[i, 0].axis("off")
        axes[i, 1].imshow(mt, cmap="gray", vmin=0, vmax=1); axes[i, 1].set_title("Points GT (1mm)", fontsize=9); axes[i, 1].axis("off")
        axes[i, 2].imshow(mp, cmap="gray", vmin=0, vmax=1); axes[i, 2].set_title("Points predits", fontsize=9); axes[i, 2].axis("off")
        ov = img.copy()
        tp = (mp > 0.5) & (mt > 0.5); fp = (mp > 0.5) & (mt < 0.5); fn = (mp < 0.5) & (mt > 0.5)
        ov[tp] = [0, 1, 0]; ov[fp] = [1, 0, 0]; ov[fn] = [0, 0, 1]
        axes[i, 3].imshow(ov); axes[i, 3].set_title("Overlay (V=TP, R=FP, B=FN)", fontsize=9); axes[i, 3].axis("off")
    plt.tight_layout(); plt.savefig(save_path, dpi=100, bbox_inches="tight"); plt.close()


# ═══════════════ Boucles d'entrainement ═══════════════
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train(); total_loss, n = 0, 0
    tm = {"dice": 0, "iou": 0, "precision": 0, "recall": 0}
    pbar = tqdm(loader, desc="  Train", leave=False)
    for images, masks in pbar:
        images, masks = images.to(device), masks.to(device)
        pred = model(images); loss = criterion(pred, masks)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        m = compute_metrics(pred, masks)
        total_loss += loss.item()
        for k in tm: tm[k] += m[k]
        n += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}", dice=f"{m['dice']:.3f}")
    return total_loss / max(n, 1), {k: v / max(n, 1) for k, v in tm.items()}

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval(); total_loss, n = 0, 0
    tm = {"dice": 0, "iou": 0, "precision": 0, "recall": 0}
    for images, masks in tqdm(loader, desc="  Val  ", leave=False):
        images, masks = images.to(device), masks.to(device)
        pred = model(images); loss = criterion(pred, masks)
        m = compute_metrics(pred, masks)
        total_loss += loss.item()
        for k in tm: tm[k] += m[k]
        n += 1
    return total_loss / max(n, 1), {k: v / max(n, 1) for k, v in tm.items()}


def _plot_training_curves(history, save_path):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    e = range(1, len(history["train_loss"]) + 1)
    a1.plot(e, history["train_loss"], "b-", label="Train")
    a1.plot(e, history["val_loss"],   "r-", label="Val")
    a1.set_xlabel("Epoch"); a1.set_ylabel("Loss"); a1.set_title("Loss"); a1.legend(); a1.grid(True, alpha=0.3)
    a2.plot(e, history["train_dice"], "b-",  label="Train Dice")
    a2.plot(e, history["val_dice"],   "r-",  label="Val Dice")
    a2.plot(e, history["train_iou"],  "b--", alpha=0.5, label="Train IoU")
    a2.plot(e, history["val_iou"],    "r--", alpha=0.5, label="Val IoU")
    a2.set_xlabel("Epoch"); a2.set_ylabel("Score"); a2.set_title("Dice & IoU"); a2.legend(); a2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.close()


def train(cfg: TrainConfig):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = os.path.join(cfg.output_dir, f"run_{timestamp}")
    os.makedirs(os.path.join(run_dir, "checkpoints"), exist_ok=True)
    os.makedirs(os.path.join(run_dir, "visualizations"), exist_ok=True)

    cfg_dict = {k: (v if isinstance(v, (int, float, bool, list, type(None))) else str(v))
                for k, v in cfg.__dict__.items()}
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(cfg_dict, f, indent=2)

    print(f"\n{'='*60}\n  Entrainement U-Net -- Detection points intersections (NPZ)\n{'='*60}")
    print(f"  Device      : {cfg.device}")
    print(f"  Resolution  : {cfg.img_height}x{cfg.img_width}")
    print(f"  Cible NPZ   : {cfg.npz_key}  (point_radius={cfg.point_radius}px, sigma={cfg.gaussian_sigma}px - GAUSSIAN)")
    print(f"  Encoder     : {cfg.encoder_name}")
    print(f"  Loss        : {cfg.loss_type}")
    print(f"  Batch size  : {cfg.batch_size}")
    print(f"  Epochs      : {cfg.num_epochs}")
    print(f"  Output      : {run_dir}\n{'='*60}\n")

    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
    device = torch.device(cfg.device)

    print("[*] Chargement des donnees...")
    print(f"  Images: {cfg.image_dir}")
    print(f"  NPZ   : {cfg.npz_dir}")

    print(f"\n  Train (sources: {cfg.train_sources}):")
    train_ds = ECGNpzDataset(cfg.image_dir, cfg.npz_dir, cfg.npz_key,
                             cfg.train_sources, cfg.img_height, cfg.img_width,
                             cfg.point_radius, cfg.gaussian_sigma, augment=True)
    print(f"  Val   (sources: {cfg.val_sources}):")
    val_ds = ECGNpzDataset(cfg.image_dir, cfg.npz_dir, cfg.npz_key,
                           cfg.val_sources, cfg.img_height, cfg.img_width,
                           cfg.point_radius, cfg.gaussian_sigma, augment=False)

    if len(train_ds) == 0:
        print("\n[ERREUR] Aucune donnee d'entrainement trouvee."); return

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)

    print("\n[*] Construction du modele...")
    model = smp.Unet(encoder_name=cfg.encoder_name, encoder_weights=cfg.encoder_weights,
                     in_channels=cfg.in_channels, classes=cfg.num_classes, activation=None).to(device)
    total = sum(p.numel() for p in model.parameters())
    print(f"  Parametres: {total:,}")

    criterion = get_loss(cfg.loss_type, cfg.bce_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=cfg.scheduler_patience, factor=cfg.scheduler_factor)

    history = {k: [] for k in ["train_loss", "val_loss", "train_dice", "val_dice", "train_iou", "val_iou", "lr"]}
    best_val_dice, no_improve = 0, 0

    print(f"\n>>> Debut de l'entrainement ({cfg.num_epochs} epochs)...\n")
    t_start = time.time()

    for epoch in range(1, cfg.num_epochs + 1):
        t0 = time.time()
        lr = optimizer.param_groups[0]["lr"]

        tl, tm = train_one_epoch(model, train_loader, criterion, optimizer, device)
        vl, vm = validate(model, val_loader, criterion, device)
        scheduler.step(vl)

        history["train_loss"].append(tl); history["val_loss"].append(vl)
        history["train_dice"].append(tm["dice"]); history["val_dice"].append(vm["dice"])
        history["train_iou"].append(tm["iou"]);   history["val_iou"].append(vm["iou"])
        history["lr"].append(lr)

        print(f"Epoch {epoch:3d}/{cfg.num_epochs} | "
              f"Train Loss: {tl:.4f}  Dice: {tm['dice']:.3f} | "
              f"Val Loss: {vl:.4f}  Dice: {vm['dice']:.3f}  IoU: {vm['iou']:.3f} | "
              f"LR: {lr:.1e} | {time.time()-t0:.1f}s")

        if vm["dice"] > best_val_dice:
            best_val_dice = vm["dice"]; no_improve = 0
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "val_dice": best_val_dice, "config": cfg_dict},
                       os.path.join(run_dir, "checkpoints", "best_model.pth"))
            print(f"  [BEST] Nouveau meilleur modele (Dice: {best_val_dice:.4f})")
        else:
            no_improve += 1

        if epoch % cfg.save_every_n_epochs == 0:
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "val_dice": vm["dice"]},
                       os.path.join(run_dir, "checkpoints", f"checkpoint_epoch{epoch:03d}.pth"))

        if epoch % cfg.log_images_every_n_epochs == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                si, sm = next(iter(val_loader))
                sp = model(si.to(device)).cpu()
                save_prediction_grid(si, sm, sp,
                    os.path.join(run_dir, "visualizations", f"epoch_{epoch:03d}.png"))

        if no_improve >= cfg.early_stop_patience:
            print(f"\n[STOP] Early stopping apres {cfg.early_stop_patience} epochs sans amelioration"); break

    total_time = time.time() - t_start
    print(f"\n{'='*60}\n  Entrainement termine en {total_time/60:.1f} minutes")
    print(f"  Meilleur Val Dice: {best_val_dice:.4f}")
    print(f"  Resultats dans: {run_dir}\n{'='*60}")
    with open(os.path.join(run_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)
    _plot_training_curves(history, os.path.join(run_dir, "training_curves.png"))
    return run_dir


In [ ]:
# Lancement de l'entrainement
train(cfg)


In [ ]:
import glob
from IPython.display import display

runs_dir = cfg.output_dir
run_dirs = sorted(glob.glob(os.path.join(runs_dir, "run_*")))
if not run_dirs:
    raise FileNotFoundError(f"Aucun run trouve dans {runs_dir}")
run_dir = run_dirs[-1]
print(f"Run selectionne : {run_dir}")

best_path = os.path.join(run_dir, "checkpoints", "best_model.pth")
ckpt_list = sorted(glob.glob(os.path.join(run_dir, "checkpoints", "*.pth")))
checkpoint_path = best_path if os.path.exists(best_path) else (ckpt_list[-1] if ckpt_list else None)
if checkpoint_path is None:
    raise FileNotFoundError("Aucun checkpoint trouve")
print(f"Checkpoint : {checkpoint_path}")

DEVICE = cfg.device
model = smp.Unet(encoder_name=cfg.encoder_name, encoder_weights=None,
                 in_channels=cfg.in_channels, classes=cfg.num_classes)
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE).eval()
print(f"Modele charge - epoch {checkpoint['epoch']} | Val Dice : {checkpoint.get('val_dice', 'N/A')}")


In [ ]:
image_dir = cfg.image_dir
val_images   = sorted([f for f in os.listdir(image_dir)
                       if f.startswith("ECG_033") and f.endswith(".webp")])
train_images = sorted([f for f in os.listdir(image_dir)
                       if (f.startswith("ECG_031") or f.startswith("ECG_032"))
                       and f.endswith(".webp")])
print(f"Train: {len(train_images)} | Val: {len(val_images)}")


In [ ]:
%matplotlib inline
SET   = "val"   # "val" ou "train"
INDEX = 394

image_list = val_images if SET == "val" else train_images
if INDEX >= len(image_list):
    raise IndexError(f"INDEX={INDEX} hors limites - {len(image_list)} images dans '{SET}'")

sample_file = image_list[INDEX]
sample_path = os.path.join(image_dir, sample_file)
npz_path    = os.path.join(cfg.npz_dir, sample_file.replace(".webp", ""))

print(f"Image : {sample_file}")
print(f"NPZ   : {npz_path}")
if not os.path.exists(npz_path):
    raise FileNotFoundError(f"NPZ introuvable : {npz_path}")

# === Image (preprocessing identique au training : /255) ===
img_pil = Image.open(sample_path).convert("RGB")
Wn, Hn = img_pil.size
img_pil = img_pil.resize((cfg.img_width, cfg.img_height), Image.BILINEAR)
img_np  = np.array(img_pil, dtype=np.float32) / 255.0

# === Mask GT depuis NPZ (meme logique que ECGNpzDataset._draw_points_mask) ===
data = load_unified(npz_path)
pts  = data.get(cfg.npz_key, np.empty((0, 2)))
mask_full = ECGNpzDataset._draw_points_mask(pts, Hn, Wn, cfg.point_radius, cfg.gaussian_sigma)
mask_pil  = Image.fromarray(mask_full).resize((cfg.img_width, cfg.img_height), Image.NEAREST)
mask_np   = np.array(mask_pil, dtype=np.float32) / 255.0

# === Inference ===
img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
with torch.no_grad():
    output = model(img_tensor)
    prediction = torch.sigmoid(output)
pred_np     = prediction.squeeze().cpu().numpy()
pred_binary = (pred_np > 0.5).astype(np.float32)

inter = (pred_binary * mask_np).sum()
dice  = (2 * inter) / (pred_binary.sum() + mask_np.sum() + 1e-8)
print(f"\nMax: {pred_np.max():.4f} | Moyenne: {pred_np.mean():.4f}")
print(f"Dice sur cette image : {dice:.4f}")

# === Overlay ===
overlay = img_np.copy()
tp = (pred_binary > 0.5) & (mask_np > 0.5)
fp = (pred_binary > 0.5) & (mask_np < 0.5)
fn = (pred_binary < 0.5) & (mask_np > 0.5)
overlay[tp] = [0.0, 1.0, 0.0]
overlay[fp] = [1.0, 0.0, 0.0]
overlay[fn] = [0.0, 0.0, 1.0]

# === Affichage ===
fig, axes = plt.subplots(1, 4, figsize=(22, 6))
fig.suptitle(f"[{SET.upper()}] {sample_file} - Dice: {dice:.4f}", fontsize=12)
axes[0].imshow(img_np);                              axes[0].set_title("Image augmentee");        axes[0].axis("off")
axes[1].imshow(mask_np,   cmap="gray", vmin=0, vmax=1); axes[1].set_title("Points GT (1mm)");        axes[1].axis("off")
axes[2].imshow(pred_np,   cmap="gray", vmin=0, vmax=1); axes[2].set_title("Prediction (sigmoid)");   axes[2].axis("off")
axes[3].imshow(overlay);                             axes[3].set_title("Overlay (V=TP, R=FP, B=FN)"); axes[3].axis("off")
plt.tight_layout(); plt.show(); plt.close()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Metriques point-based : nb de points detectes + deviation en pixels
# (proposition Dr Menet : "nb de points + deviation par rapport au pixel")
# Affichage 100% inline - aucun fichier ecrit sur disque.
# ═══════════════════════════════════════════════════════════════════
import io
import cv2
from scipy.spatial import cKDTree
from IPython.display import display, Image as IPyImage

%matplotlib inline
SET            = "val"   # "val" ou "train"
INDEX          = 8
MAX_MATCH_DIST = 15      # px (resolution native) : distance max pour considerer un match valide
THRESHOLD      = 0.5     # seuil sur sigmoid pour binarisation

image_list = val_images if SET == "val" else train_images
sample_file = image_list[INDEX]
sample_path = os.path.join(image_dir, sample_file)
npz_path    = os.path.join(cfg.npz_dir, sample_file.replace(".webp", ""))

# === Image en resolution native + version resize pour le modele ===
img_pil_native = Image.open(sample_path).convert("RGB")
Wn, Hn = img_pil_native.size
img_pil = img_pil_native.resize((cfg.img_width, cfg.img_height), Image.BILINEAR)
img_np  = np.array(img_pil, dtype=np.float32) / 255.0

# === GT : coordonnees brutes des intersections (resolution native) ===
data         = load_unified(npz_path)
gt_pts_full  = data.get(cfg.npz_key, np.empty((0, 2)))
if len(gt_pts_full):
    valid     = (gt_pts_full[:, 0] >= 0) & (gt_pts_full[:, 0] < Wn) & \
                (gt_pts_full[:, 1] >= 0) & (gt_pts_full[:, 1] < Hn)
    gt_pts    = gt_pts_full[valid]
else:
    gt_pts    = gt_pts_full

# === Inference + upscale a la resolution native ===
img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
with torch.no_grad():
    pred_sigmoid = torch.sigmoid(model(img_tensor)).squeeze().cpu().numpy()
pred_native = cv2.resize(pred_sigmoid, (Wn, Hn), interpolation=cv2.INTER_LINEAR)
pred_binary = (pred_native > THRESHOLD).astype(np.uint8)

# === Extraction des centres des blobs predits via CENTROIDE PONDERE ===
# Avec la heatmap gaussienne, on utilise le centroide pondere par l'intensite sigmoid
# (et non le centroide simple des pixels binarises) pour obtenir une precision sub-pixel.
# Etape 1 : seuil bas pour separer les blobs distincts.
SEPARATION_THRESHOLD = 0.3
sep_binary = (pred_native > SEPARATION_THRESHOLD).astype(np.uint8)
num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(sep_binary, connectivity=8)
MIN_BLOB_AREA = 3
pred_pts_list = []
for lab in range(1, num_labels):
    if stats[lab, cv2.CC_STAT_AREA] < MIN_BLOB_AREA:
        continue
    blob_mask = (labels == lab)
    weights = pred_native[blob_mask]      # intensite sigmoid (continue) dans le blob
    yy, xx = np.where(blob_mask)
    w_sum = weights.sum()
    if w_sum < 1e-6:
        continue
    cx = float((xx * weights).sum() / w_sum)
    cy = float((yy * weights).sum() / w_sum)
    pred_pts_list.append((cx, cy))
pred_pts = np.asarray(pred_pts_list) if pred_pts_list else np.empty((0, 2))

# === Matching greedy au plus proche voisin (avec seuil de distance) ===
nb_gt   = len(gt_pts)
nb_pred = len(pred_pts)
matched_pred_idx, matched_gt_idx, matched_dists = [], [], []

if nb_gt > 0 and nb_pred > 0:
    tree = cKDTree(gt_pts)
    dists, gt_idxs = tree.query(pred_pts, k=1)
    order = np.argsort(dists)
    used_gt = set()
    for pi in order:
        d, gi = dists[pi], int(gt_idxs[pi])
        if d <= MAX_MATCH_DIST and gi not in used_gt:
            matched_pred_idx.append(pi)
            matched_gt_idx.append(gi)
            matched_dists.append(d)
            used_gt.add(gi)

nb_matched = len(matched_dists)

# === Calcul des metriques ===
recall    = nb_matched / nb_gt   if nb_gt   > 0 else 0.0
precision = nb_matched / nb_pred if nb_pred > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

if matched_dists:
    arr_d      = np.asarray(matched_dists)
    dev_mean   = arr_d.mean()
    dev_median = np.median(arr_d)
    dev_max    = arr_d.max()
    dev_p95    = np.percentile(arr_d, 95)
else:
    dev_mean = dev_median = dev_max = dev_p95 = float("nan")

# === Preparation des points pour visualisation ===
matched_pred_set = set(matched_pred_idx)
matched_gt_set   = set(matched_gt_idx)
fn_pts = gt_pts[[i for i in range(nb_gt)   if i not in matched_gt_set]]   if nb_gt   else np.empty((0, 2))
fp_pts = pred_pts[[i for i in range(nb_pred) if i not in matched_pred_set]] if nb_pred else np.empty((0, 2))
match_pred_pts = pred_pts[matched_pred_idx] if matched_pred_idx else np.empty((0, 2))
match_gt_pts   = gt_pts[matched_gt_idx]     if matched_gt_idx   else np.empty((0, 2))

# === Categorisation pixel : exact match (vert) vs separes (blanc + rouge) ===
nb_exact = 0
if nb_matched > 0:
    gt_xi = np.clip(np.round(match_gt_pts[:, 0]).astype(int),   0, Wn - 1)
    gt_yi = np.clip(np.round(match_gt_pts[:, 1]).astype(int),   0, Hn - 1)
    pr_xi = np.clip(np.round(match_pred_pts[:, 0]).astype(int), 0, Wn - 1)
    pr_yi = np.clip(np.round(match_pred_pts[:, 1]).astype(int), 0, Hn - 1)
    same_pixel = (gt_xi == pr_xi) & (gt_yi == pr_yi)
    nb_exact = int(same_pixel.sum())

# === Construction du tableau pixel-perfect (1 px = 1 px) en pleine taille ===
viz_full = np.zeros((Hn, Wn, 3), dtype=np.uint8)
if nb_matched > 0:
    diff_mask = ~same_pixel
    viz_full[gt_yi[diff_mask], gt_xi[diff_mask]] = (255, 255, 255)
    viz_full[pr_yi[diff_mask], pr_xi[diff_mask]] = (255,   0,   0)
    viz_full[gt_yi[same_pixel], gt_xi[same_pixel]] = (0, 255, 0)

# === Affichage texte des metriques ===
print("=" * 60)
print(f"Image : {sample_file}")
print(f"Resolution native : {Wn}x{Hn}")
print(f"Seuil sigmoid : {THRESHOLD}  |  Dist max match : {MAX_MATCH_DIST} px")
print("=" * 60)
print(f"\n[POINTS]")
print(f"  Points GT (vraie verite)         : {nb_gt}")
print(f"  Points predits (apres seuil)     : {nb_pred}")
print(f"  Points matches (TP)              : {nb_matched}")
print(f"  Points GT manques (FN)           : {nb_gt - nb_matched}")
print(f"  Predictions sans GT (FP)         : {nb_pred - nb_matched}")
print(f"  Matches sur le MEME pixel (vert) : {nb_exact}  ({100*nb_exact/max(nb_matched,1):.1f}%)")
print(f"  Matches a 1-2 px (blanc+rouge)   : {nb_matched - nb_exact}  ({100*(nb_matched-nb_exact)/max(nb_matched,1):.1f}%)")
print(f"\n[TAUX DE DETECTION]")
print(f"  Recall    = TP/GT     : {recall*100:6.2f}%")
print(f"  Precision = TP/Pred   : {precision*100:6.2f}%")
print(f"  F1        = harmonique: {f1*100:6.2f}%")
print(f"\n[DEVIATION PIXEL (resolution native)]")
print(f"  Moyenne   : {dev_mean:.2f} px")
print(f"  Mediane   : {dev_median:.2f} px")
print(f"  P95       : {dev_p95:.2f} px")
print(f"  Maximum   : {dev_max:.2f} px")
print("=" * 60)

img_native_arr = np.array(img_pil_native)

# === Figure 1 : 2 sous-graphes overview (image + scatters) ===
fig, axes = plt.subplots(1, 2, figsize=(22, 10))
fig.suptitle(f"[{SET.upper()}] {sample_file}  -  Recall: {recall*100:.1f}%  |  Dev moyenne: {dev_mean:.2f} px",
             fontsize=14)

# Subplot 1 : tous les points GT (bleu) vs Predits (rouge) sur l'image
axes[0].imshow(img_native_arr)
axes[0].set_title(f"Tous les points : GT (bleu, {nb_gt}) vs Predits (rouge, {nb_pred})", fontsize=12)
if nb_gt > 0:
    axes[0].scatter(gt_pts[:, 0],   gt_pts[:, 1],   s=10, c="blue", alpha=0.5, label=f"GT ({nb_gt})")
if nb_pred > 0:
    axes[0].scatter(pred_pts[:, 0], pred_pts[:, 1], s=10, c="red",  alpha=0.5, label=f"Predits ({nb_pred})")
axes[0].legend(loc="upper right"); axes[0].axis("off")

# Subplot 2 : classification TP/FN/FP sur l'image
axes[1].imshow(img_native_arr)
axes[1].set_title(f"Vert=TP ({nb_matched})  |  Bleu=FN ({len(fn_pts)})  |  Rouge=FP ({len(fp_pts)})", fontsize=12)
if len(match_pred_pts):
    axes[1].scatter(match_pred_pts[:, 0], match_pred_pts[:, 1], s=14, c="lime", alpha=0.9, label=f"TP ({nb_matched})")
if len(fn_pts):
    axes[1].scatter(fn_pts[:, 0], fn_pts[:, 1], s=14, c="blue", alpha=0.9, label=f"FN ({len(fn_pts)})")
if len(fp_pts):
    axes[1].scatter(fp_pts[:, 0], fp_pts[:, 1], s=14, c="red",  alpha=0.9, label=f"FP ({len(fp_pts)})")
axes[1].legend(loc="upper right"); axes[1].axis("off")

plt.tight_layout(); plt.show(); plt.close()

# === Figure 2 : Pixels reels (1 px = 1 px) - IMAGE COMPLETE, AFFICHAGE EN MEMOIRE ===
# Pas de sauvegarde sur disque : on encode le PNG en RAM dans un BytesIO et on
# l'envoie a IPython.display.Image qui le rend a sa resolution native sans downscale.
print(f"\nPixels reels (image complete {Wn}x{Hn}, 1 px = 1 px) :")
print(f"  Blanc = GT seul ({nb_matched - nb_exact})")
print(f"  Rouge = Predit seul ({nb_matched - nb_exact})")
print(f"  Vert  = meme pixel ({nb_exact})")

buf_viz = io.BytesIO()
Image.fromarray(viz_full).save(buf_viz, format="PNG")
display(IPyImage(data=buf_viz.getvalue(), format="png"))

# === Figure 3 : zooms automatiques sur 6 cas (du meilleur au pire) ===
if nb_matched > 0:
    sorted_idx = np.argsort(arr_d)
    n = len(sorted_idx)
    interesting = {
        "min":     int(sorted_idx[0]),
        "p20":     int(sorted_idx[int(n * 0.20)]),
        "p40":     int(sorted_idx[int(n * 0.40)]),
        "p60":     int(sorted_idx[int(n * 0.60)]),
        "p80":     int(sorted_idx[int(n * 0.80)]),
        "max":     int(sorted_idx[-1]),
    }
    ZOOM_RADIUS = 30

    fig_zoom, ax_zoom = plt.subplots(2, 3, figsize=(18, 12))
    ax_zoom = ax_zoom.ravel()
    for ax, (label, idx) in zip(ax_zoom, interesting.items()):
        gx, gy = match_gt_pts[idx]
        px, py = match_pred_pts[idx]
        cx, cy = int((gx + px) / 2), int((gy + py) / 2)
        zx0 = max(0, cx - ZOOM_RADIUS); zx1 = min(Wn, cx + ZOOM_RADIUS)
        zy0 = max(0, cy - ZOOM_RADIUS); zy1 = min(Hn, cy + ZOOM_RADIUS)
        crop = img_native_arr[zy0:zy1, zx0:zx1].copy()
        ax.imshow(crop, interpolation="nearest")
        ax.scatter(gx - zx0, gy - zy0, s=150, c="white", marker="x", linewidths=2, label="GT")
        ax.scatter(px - zx0, py - zy0, s=150, c="red",   marker="+", linewidths=2, label="Predit")
        ax.set_title(f"{label} (deviation = {arr_d[idx]:.2f} px)", fontsize=11)
        ax.legend(loc="lower right", fontsize=9)
        ax.axis("off")
    fig_zoom.suptitle(f"Zooms (rayon {ZOOM_RADIUS}px) : croix blanche=GT, croix rouge=Predit", fontsize=12)
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
# Test du modele sur image reelle (data/output_real/)
import os, glob, cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import segmentation_models_pytorch as smp

%matplotlib inline

# Verification stricte : comparer les poids en memoire vs disk
import glob, os, torch
runs = sorted(glob.glob(os.path.join(cfg.output_dir, "run_*")))
ckpt_path = os.path.join(runs[-1], "checkpoints", "best_model.pth")
ckpt_disk = torch.load(ckpt_path, map_location='cpu', weights_only=False)

# Recuperer un poids specifique en memoire vs disk
weight_name = list(ckpt_disk["model_state_dict"].keys())[5]   # un poids arbitraire
w_mem  = model.state_dict()[weight_name].cpu().flatten()[:5]
w_disk = ckpt_disk["model_state_dict"][weight_name].flatten()[:5]
print(f"Weight tested      : {weight_name}")
print(f"In memory (5 vals) : {w_mem.numpy()}")
print(f"On disk    (5 vals): {w_disk.numpy()}")
print(f"MATCH              : {torch.allclose(w_mem, w_disk)}")
print(f"Ckpt epoch         : {ckpt_disk['epoch']}")
print(f"Ckpt val_dice      : {ckpt_disk.get('val_dice', 'N/A')}")

# === A modifier au besoin ===
REAL_ROOT = r"C:\Users\v\Desktop\ECGPerturb-main\data\output_real"
SUBFOLDER = "photos_crumbles"   # ou None pour scanner tout
INDEX = 1                        # quel image dans la liste
OVERLAY_COLOR = (255, 0, 0)
OVERLAY_ALPHA = 0.55

# === Liste des images reelles ===
if SUBFOLDER:
    search = os.path.join(REAL_ROOT, SUBFOLDER, "*")
else:
    search = os.path.join(REAL_ROOT, "**", "*")
real_files = sorted([f for f in glob.glob(search, recursive=True)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp'))])
print(f"{len(real_files)} images reelles trouvees")

if INDEX >= len(real_files):
    raise IndexError(f"INDEX={INDEX} hors limites (max {len(real_files)-1})")
img_path = real_files[INDEX]
print(f"Image : {img_path}")

# === Chargement du modele (si pas deja en memoire) ===
try:
    model
except NameError:
    runs = sorted(glob.glob(os.path.join(cfg.output_dir, "run_*")))
    if not runs:
        raise FileNotFoundError(f"Aucun run dans {cfg.output_dir}")
    ckpt_path = os.path.join(runs[-1], "checkpoints", "best_model.pth")
    model = smp.Unet(encoder_name=cfg.encoder_name, encoder_weights=None,
                     in_channels=cfg.in_channels, classes=cfg.num_classes)
    ckpt = torch.load(ckpt_path, map_location=cfg.device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(cfg.device).eval()
    print(f"Modele charge - epoch {ckpt['epoch']} | val_dice = {ckpt.get('val_dice', 'N/A')}")

# === Preprocessing (identique au training : /255, pas d'imagenet norm) ===
img_pil = Image.open(img_path).convert("RGB")
W_orig, H_orig = img_pil.size
print(f"Resolution native : {W_orig}x{H_orig}")

img_resized = img_pil.resize((cfg.img_width, cfg.img_height), Image.BILINEAR)
img_np = np.array(img_resized, dtype=np.float32) / 255.0
img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).float().to(cfg.device)

# === Inference ===
with torch.no_grad():
    pred_sigmoid = torch.sigmoid(model(img_tensor)).squeeze().cpu().numpy()

pred_full = cv2.resize(pred_sigmoid, (W_orig, H_orig), interpolation=cv2.INTER_LINEAR)
pred_binary = (pred_full > 0.5).astype(np.uint8) * 255

print(f"Heatmap : max={pred_sigmoid.max():.3f}  mean={pred_sigmoid.mean():.4f}")
print(f"Pixels positifs (seuil 0.5) : {(pred_binary>0).sum()} ({100*(pred_binary>0).sum()/pred_binary.size:.2f}%)")

# === Superposition ===
img_arr = np.array(img_pil)
overlay = img_arr.copy().astype(np.float32)
mask_bool = pred_binary > 127
color = np.array(OVERLAY_COLOR, dtype=np.float32)
overlay[mask_bool] = (1 - OVERLAY_ALPHA) * overlay[mask_bool] + OVERLAY_ALPHA * color
overlay = overlay.clip(0, 255).astype(np.uint8)

# === Affichage 4-up ===
fig, axes = plt.subplots(1, 4, figsize=(28, 8))
fig.suptitle(f"Inference reelle : {os.path.basename(img_path)}", fontsize=12)
axes[0].imshow(img_arr);                                                          axes[0].set_title(f"Image reelle ({W_orig}x{H_orig})");           axes[0].axis("off")
axes[1].imshow(pred_full, cmap="hot", vmin=0, vmax=max(pred_full.max(), 1e-6));   axes[1].set_title(f"Sigmoid (max={pred_full.max():.2f})");        axes[1].axis("off")
axes[2].imshow(pred_binary, cmap="gray", vmin=0, vmax=255);                       axes[2].set_title("Mask binarise (seuil 0.5)");                   axes[2].axis("off")
axes[3].imshow(overlay);                                                          axes[3].set_title("Superposition");                               axes[3].axis("off")
plt.tight_layout(); plt.show()